# 📘 Modèle Pyomo généré automatiquement

## 📦 Imports

In [1]:
from pyomo.environ import *
from pyomo.opt import SolverFactory
import pandas as pd

## 🔹 Model

In [2]:
from pyomo.environ import *

model = ConcreteModel()

## 🔹 Sets

In [3]:
model.PRODUITS = Set(initialize=['P1', 'P2'])
model.USINES = Set(initialize=['U1', 'U2', 'U3'])
model.PRODS_USINES = Set(dimen=2, initialize=[(i,j) for i in model.USINES for j in model.PRODUITS])

## 🔹 Parameters

In [4]:
model.gain = Param(model.PRODUITS, initialize={'P1': 3.0, 'P2': 5.0}, within=NonNegativeReals)
model.disp = Param(model.USINES, initialize={'U1': 4.0, 'U2': 12.0, 'U3': 18.0}, within=NonNegativeReals)
model.utilisation = Param(model.USINES, model.PRODUITS, initialize={('U1', 'P1'): 1.0, ('U1', 'P2'): 0.0, ('U2', 'P1'): 0.0, ('U2', 'P2'): 2.0, ('U3', 'P1'): 3.0, ('U3', 'P2'): 2.0}, within=NonNegativeReals)

## 🔹 Variables

In [5]:
model.x = Var(model.PRODUITS, domain=NonNegativeReals)

## 🔹 Constraints

In [6]:
model.c_for_0 = ConstraintList()
for u in model.USINES:
    model.c_for_0.add(sum(model.utilisation[u,p] * model.x[p] for p in model.PRODUITS) <= model.disp[u])

## 🔹 Objective

In [7]:
model.obj = Objective(expr=sum(model.gain[p] * model.x[p] for p in model.PRODUITS), sense=maximize)

## ⚙️ Résolution du modèle

In [8]:
solver = SolverFactory('highs')
result = solver.solve(model, tee=True)

print('✅ Solver status:', result.solver.status)
print('✅ Termination condition:', result.solver.termination_condition)

✅ Solver status: ok
✅ Termination condition: optimal


## 🎯 Valeur de la fonction objective

In [9]:
for obj in model.component_objects(Objective, active=True):
    print(f'Objectif: {obj.name}')
    print(f'Valeur optimale: {obj():.4f}')
    print(f'Sens: {"Minimisation" if obj.sense == minimize else "Maximisation"}')

Objectif: obj
Valeur optimale: 36.0000
Sens: Maximisation


## 📊 Valeurs optimales des variables

In [10]:
# Extraction des résultats dans un DataFrame
results_data = []
for v in model.component_objects(Var, active=True):
    for index in v:
        results_data.append({
            'Variable': v.name,
            'Index': str(index) if index != None else '-',
            'Valeur': v[index].value
        })

df_results = pd.DataFrame(results_data)
# Filtrer les valeurs non-nulles pour plus de clarté
df_results = df_results[df_results['Valeur'].notna()]
df_results = df_results[df_results['Valeur'] != 0]
df_results.style.format({'Valeur': '{:.4f}'}).set_caption('Variables de décision optimales')

,Variable,Index,Valeur
0,x,P1,2.0000
1,x,P2,6.0000
